## Supplementary Table 6

Enviorment: env 1

In [1]:
import pandas as pd
import pickle
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

In [2]:
dpi = 300
font_size = 12
fig_size = 5
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
plt.rcParams["font.family"] = "DejaVu Sans"

### load NYU data

#### fmf

In [3]:
fmf_probs = pd.read_csv("/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/data/fmf_pe_results_104.csv")
fmf_probs["id"] = [id.lower() for id in fmf_probs["patient_id"]]
fmf_probs["first_preg"] = [int(val == "Nulliparous") for val in fmf_probs["input_obhist"]]
fmf_probs["prev_pec"] = [int(val == "Parous, prev PE") for val in fmf_probs["input_obhist"]]

In [4]:
fmf_probs["chtn"] = fmf_probs["input_hyp"].astype(int)

#### maternal information

In [5]:
nyu_clinical = pd.read_excel("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/Visionary AI Participant Tracker (1).xlsx")
nyu_clinical["id"] = [id.lower() for id in nyu_clinical["Study ID"].fillna("nan")]
nyu_clinical["aspirin"] = [int("yes" in str(val).lower()) for val in nyu_clinical["Recommended Aspirin?"]]

#### retinal information

In [6]:
nyu_path = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/nyu/nyu_rg_delivered_testset/"
all_probs = []
all_shap_list = []
for i in range(1, 6):
    path = nyu_path + f"nyu_79_no_repeats_med_pec_pw{i}/meta_all_0.5_oof_bls_pw{i}/XGBoost/"
    name = os.listdir(path)[0]
    with open(path + name, "rb") as f:
        data = pickle.load(f)
        for id in data[5].keys():
            all_shap_list.append({"pw": i,"id": id, "shap": data[5][id]})
    all_probs.append(data[1])
all_probs = pd.DataFrame(all_probs)

In [7]:
with open("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/nyu_paper_cases.pkl", "rb") as f:
    pec = pickle.load(f)

In [8]:
retinal_probs = pd.DataFrame(all_probs.mean(), columns=["retinal"])
retinal_probs["id"] = retinal_probs.index
retinal_probs = retinal_probs.reset_index(drop=True)
retinal_probs = retinal_probs.merge(fmf_probs[["id", "pe_before_delivery_percent", "cohort", "first_preg", "prev_pec", "input_age", "chtn"]], how="outer")
retinal_probs["label"] = (retinal_probs["cohort"] == "case").astype(int)
retinal_probs = retinal_probs.dropna()

#### IDs

In [9]:
with open("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/nyu_evaluation.pkl", "rb") as f:
    eval_data = pickle.load(file=f)
ghtn_cases = eval_data["ghtn"]
chtn_cases = eval_data["chtn"]
final_exclusions = eval_data["ex"]

In [10]:
ids = list(set(retinal_probs["id"].values) - set(final_exclusions))

In [11]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, roc_auc_score
import matplotlib.pyplot as plt


def bootstrap_roc_pr_and_compare(
    df,
    label_col,
    score_cols_dict,
    threshold_dict,
    n_bootstraps=10000,
    seed=8,
    alpha=0.05,
    n_points=100
):
    """
    gets bootstrapped values to compare
    """
    rng = np.random.RandomState(seed)

    n = len(df)
    labels = df[label_col].values

    # get indices for cases and controls separately
    case_indices = np.where(labels == 1)[0]
    control_indices = np.where(labels == 0)[0]
    n_cases = len(case_indices)
    n_controls = len(control_indices)
    print("cases", n_cases)
    print("controls", n_controls)

    bootstrap_indices = []
    for _ in range(n_bootstraps):
        # sample cases with replacement (same count as original)
        boot_cases = rng.choice(case_indices, size=n_cases, replace=True)
        # sample controls with replacement (same count as original)
        boot_controls = rng.choice(control_indices, size=n_controls, replace=True)
        # combine
        boot_idx = np.concatenate([boot_cases, boot_controls])
        bootstrap_indices.append(boot_idx)

    # compute metrics
    model_names = list(score_cols_dict.keys())

    model_roc_data = {}
    model_pr_data = {}
    model_boot_aucs = {}
    model_boot_aps = {}
    model_boot_threshold_metrics = {}

    mean_fpr = np.linspace(0, 1, n_points)
    mean_recall = np.linspace(0, 1, n_points)

    def _compute_threshold_metrics(boot_labels, boot_scores, thresh, prevalence=0.08):
        """
        get threshold metrics
        """
        preds = (boot_scores >= thresh).astype(int)
        tp = np.sum((preds == 1) & (boot_labels == 1))
        fp = np.sum((preds == 1) & (boot_labels == 0))
        tn = np.sum((preds == 0) & (boot_labels == 0))
        fn = np.sum((preds == 0) & (boot_labels == 1))

        # Standard metrics
        ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan  # sensitivity / recall
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        tnr = tn / (tn + fp) if (tn + fp) > 0 else np.nan  # specificity

        # F1 score
        if ppv is not np.nan and tpr is not np.nan and (ppv + tpr) > 0:
            f1 = 2 * (ppv * tpr) / (ppv + tpr)
        else:
            f1 = np.nan

        # balanced accuracy
        if tpr is not np.nan and tnr is not np.nan:
            balanced_acc = (tpr + tnr) / 2
        else:
            balanced_acc = np.nan

        # prevalence-adjusted PPV and NPV using Bayes' theorem
        # PPV_adj = (sensitivity * prevalence) / ((sensitivity * prevalence) + ((1 - specificity) * (1 - prevalence)))
        # NPV_adj = (specificity * (1 - prevalence)) / ((specificity * (1 - prevalence)) + ((1 - sensitivity) * prevalence))
        if tpr is not np.nan and fpr is not np.nan:
            ppv_numerator = tpr * prevalence
            ppv_denominator = (tpr * prevalence) + (fpr * (1 - prevalence))
            ppv_adj = ppv_numerator / ppv_denominator if ppv_denominator > 0 else np.nan
        else:
            ppv_adj = np.nan

        if tnr is not np.nan and tpr is not np.nan:
            npv_numerator = tnr * (1 - prevalence)
            npv_denominator = (tnr * (1 - prevalence)) + ((1 - tpr) * prevalence)
            npv_adj = npv_numerator / npv_denominator if npv_denominator > 0 else np.nan
        else:
            npv_adj = np.nan

        return {
            "ppv": ppv,
            "tpr": tpr,
            "fpr": fpr,
            "npv": npv,
            "f1": f1,
            "ba": balanced_acc,
            "pa_ppv": ppv_adj,
            "pa_npv": npv_adj,
        }

    for model_name, score_col in score_cols_dict.items():
        scores = df[score_col].values
        thresh = threshold_dict[model_name]
        tprs = []
        aucs_list = []
        precisions = []
        aps_list = []
        threshold_metrics = {"ppv": [], "tpr": [], "fpr": [], "npv": [], "f1": [], "ba": [], "pa_ppv": [], "pa_npv": []}

        for idx in bootstrap_indices:
            boot_labels = labels[idx]
            boot_scores = scores[idx]

            if len(np.unique(boot_labels)) < 2:
                continue

            # ROC
            fpr_arr, tpr_arr, _ = roc_curve(boot_labels, boot_scores)
            interp_tpr = np.interp(mean_fpr, fpr_arr, tpr_arr)
            interp_tpr[0] = 0.0
            tprs.append(interp_tpr)
            aucs_list.append(auc(fpr_arr, tpr_arr))

            # PR
            precision, recall, _ = precision_recall_curve(boot_labels, boot_scores)
            precision_sorted = np.flip(precision)
            recall_sorted = np.flip(recall)
            interp_prec = np.interp(mean_recall, recall_sorted, precision_sorted)
            precisions.append(interp_prec)
            aps_list.append(average_precision_score(boot_labels, boot_scores))

            # threshold metrics (per-model threshold)
            t_metrics = _compute_threshold_metrics(boot_labels, boot_scores, thresh)
            for key in threshold_metrics:
                threshold_metrics[key].append(t_metrics[key])

        model_roc_data[model_name] = (mean_fpr, np.array(tprs), np.array(aucs_list))
        model_pr_data[model_name] = (mean_recall, np.array(precisions), np.array(aps_list))
        model_boot_aucs[model_name] = np.array(aucs_list)
        model_boot_aps[model_name] = np.array(aps_list)
        model_boot_threshold_metrics[model_name] = {
            key: np.array([v for v in vals if not np.isnan(v)])
            for key, vals in threshold_metrics.items()
        }

    # roc
    for model_name in model_names:
        mean_fpr_arr, tprs, aucs_arr = model_roc_data[model_name]
        mean_tpr = np.mean(tprs, axis=0)
        mean_tpr[-1] = 1.0
        mean_auc = np.mean(aucs_arr)
        ci_low = np.percentile(aucs_arr, 2.5)
        ci_high = np.percentile(aucs_arr, 97.5)
        print(f"{model_name} (AUC = {mean_auc:.2f} [{ci_low:.2f}-{ci_high:.2f}])")

    print()
    
    # pr
    for model_name in model_names:
        mean_recall_arr, precisions, aps_arr = model_pr_data[model_name]
        mean_prec = np.mean(precisions, axis=0)
        mean_ap = np.mean(aps_arr)
        ci_low = np.percentile(aps_arr, 2.5)
        ci_high = np.percentile(aps_arr, 97.5)
        print(f"{model_name} (AP = {mean_ap:.2f} [{ci_low:.2f}-{ci_high:.2f}])")

    # threshold metrics
    threshold_results = []
    for model_name in model_names:
        metrics = model_boot_threshold_metrics[model_name]
        thresh = threshold_dict[model_name]
        row = {"Model": model_name, "Threshold": thresh}
        for metric_name in ["ppv", "tpr", "fpr", "npv", "ba", "f1", "pa_ppv", "pa_npv"]:
            vals = metrics[metric_name]
            if len(vals) > 0:
                row[f"{metric_name.upper()}_mean"] = np.mean(vals)
                row[f"{metric_name.upper()}_CI"] = f"[{np.percentile(vals, 2.5):.4f}, {np.percentile(vals, 97.5):.4f}]"
            else:
                row[f"{metric_name.upper()}_mean"] = np.nan
                row[f"{metric_name.upper()}_CI"] = "N/A"
        threshold_results.append(row)

    threshold_df = pd.DataFrame(threshold_results)

    print("\n\n" + "=" * 80)
    print("threshold metrics")
    print("=" * 80)
    for _, row in threshold_df.iterrows():
        print(f"\n  {row['Model']} (threshold = {row['Threshold']})")
        print(f"    PPV (Precision):   {row['PPV_mean']:.2f}  95% CI: {row['PPV_CI']}")
        print(f"    TPR (Sensitivity): {row['TPR_mean']:.2f}  95% CI: {row['TPR_CI']}")
        print(f"    FPR:               {row['FPR_mean']:.2f}  95% CI: {row['FPR_CI']}")
        print(f"    NPV:               {row['NPV_mean']:.2f}  95% CI: {row['NPV_CI']}")
        print(f"    F1:               {row['F1_mean']:.2f}  95% CI: {row['F1_CI']}")
        print(f"    BA:               {row['BA_mean']:.2f}  95% CI: {row['BA_CI']}")
        print(f"    PA_PPV:               {row['PA_PPV_mean']:.2f}  95% CI: {row['PA_PPV_CI']}")
        print(f"    PA_NPV:               {row['PA_NPV_mean']:.2f}  95% CI: {row['PA_NPV_CI']}")

In [12]:
import random
import os

In [ ]:
bs_ids = ids

In [14]:
df = retinal_probs[retinal_probs["id"].isin(bs_ids)].dropna().copy()
df["combo"] = (
    df["prev_pec"] * 0.35
    + 0.15 * (df["first_preg"] * (df["input_age"] > 37).astype(int))
)

# define score columns for each model
df["score_both"] = df["pe_before_delivery_percent"] / 100 + df["retinal"] + df["combo"]
df["score_retinal"] = df["retinal"] + df["combo"]
df["score_fmf"] = df["pe_before_delivery_percent"] / 100
len(df)

66

In [15]:
SEED = 8
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

bootstrap_roc_pr_and_compare(
    df=df,
    label_col="label",
    score_cols_dict={
        "Retinal": "score_retinal",
        "FMF": "score_fmf",
    },
    threshold_dict={
        "Retinal": 0.53,
        "FMF": 0.1,
    },
    n_bootstraps=10000,
    seed=SEED,
    alpha=0.05,
)

cases 13
controls 53
Retinal (AUC = 0.80 [0.62-0.95])
FMF (AUC = 0.73 [0.58-0.86])

Retinal (AP = 0.69 [0.47-0.88])
FMF (AP = 0.51 [0.30-0.72])


threshold metrics

  Retinal (threshold = 0.53)
    PPV (Precision):   0.65  95% CI: [0.4000, 0.9000]
    TPR (Sensitivity): 0.54  95% CI: [0.2308, 0.7712]
    FPR:               0.08  95% CI: [0.0189, 0.1509]
    NPV:               0.89  95% CI: [0.8361, 0.9467]
    F1:               0.58  95% CI: [0.3333, 0.8000]
    BA:               0.73  95% CI: [0.5878, 0.8665]
    PA_PPV:               0.41  95% CI: [0.1912, 0.7614]
    PA_NPV:               0.96  95% CI: [0.9350, 0.9804]

  FMF (threshold = 0.1)
    PPV (Precision):   0.51  95% CI: [0.1667, 0.8571]
    TPR (Sensitivity): 0.31  95% CI: [0.0769, 0.5385]
    FPR:               0.08  95% CI: [0.0189, 0.1509]
    NPV:               0.85  95% CI: [0.8000, 0.8966]
    F1:               0.38  95% CI: [0.1111, 0.6364]
    BA:               0.62  95% CI: [0.4913, 0.7511]
    PA_PPV:            